In [1]:
# Instalação no servidor local
# pip install langchain langchain-community sentence-transformers faiss-cpu pymupdf tqdm pandas

# Instalação diretamente no google colab
# !pip install -q langchain langchain-community sentence-transformers faiss-cpu pymupdf tqdm pandas

In [2]:
import os
import re
import pickle
import shutil
import json
import gc

import pymupdf
import pandas as pd

from tqdm import tqdm
from collections import Counter

from langchain_core.documents import Document
from langchain_community.storage import InMemoryStore
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.retrievers import ParentDocumentRetriever

ImportError: cannot import name 'InMemoryStore' from 'langchain_community.storage' (/home/avelar/miniconda3/envs/colab-server/lib/python3.11/site-packages/langchain_community/storage/__init__.py)

In [ ]:
DOCS_PATH = "documentos/"
VECTORSTORE_PATH = "faiss_final_index"
DOCSTORE_PATH = "final_docstore"

MODEL_EMBEDDING_NAME = "BAAI/bge-m3"
DEVICE = "cuda"

PARENT_CHUNK_SIZE = 1500
PARENT_CHUNK_OVERLAP = 200

CHILD_CHUNK_SIZE = 400
CHILD_CHUNK_OVERLAP = 50

In [ ]:
def cleanup_old_index(vectorstore_path, docstore_path):
    for path in [vectorstore_path, docstore_path]:
        if os.path.exists(path):
            shutil.rmtree(path)

    os.makedirs(vectorstore_path, exist_ok=True)
    os.makedirs(docstore_path, exist_ok=True)

    print("✅ Diretórios limpos e recriados")

In [ ]:
def load_and_clean_definitively(folder_path: str) -> list:
    pdf_files = sorted([f for f in os.listdir(folder_path) if f.endswith(".pdf")])

    documents = []

    for filename in tqdm(pdf_files, desc="Processando PDFs"):
        file_path = os.path.join(folder_path, filename)
        cleaned_pages = []

        with pymupdf.open(file_path) as doc:
            page_count = len(doc)

            for page in doc:
                page_height = page.rect.height
                margin_top = page_height * 0.08
                margin_bottom = page_height * 0.92

                blocks = page.get_text("blocks")
                valid_blocks = [b for b in blocks if margin_top < b[1] < margin_bottom]
                valid_blocks.sort(key=lambda b: b[1])

                page_text = " ".join([b[4].replace('\n', ' ') for b in valid_blocks])
                page_text = re.sub(r'\s+', ' ', page_text).strip()

                cleaned_pages.append(page_text)

        full_text = " ".join(cleaned_pages)
        full_text = re.sub(r'\s+', ' ', full_text).strip()

        documents.append(
            Document(
                page_content=full_text,
                metadata={
                    "source": filename,
                    "file_name": filename,
                    "page_count": page_count,
                    "char_count": len(full_text),
                }
            )
        )

    print(f"\n✅ Total de documentos: {len(documents)}")
    return documents

In [ ]:
def build_page_count_table(documents):
    page_counts = [doc.metadata["page_count"] for doc in documents]

    counter = Counter(page_counts)
    sorted_pages = sorted(counter.keys())

    df = pd.DataFrame(
        [
            sorted_pages,
            [counter[p] for p in sorted_pages]
        ],
        index=["qtd_paginas", "qtd_documentos"]
    )

    return df

In [ ]:
def build_index(documents):
    print("🔧 Inicializando embeddings...")

    embeddings = HuggingFaceEmbeddings(
        model_name=MODEL_EMBEDDING_NAME,
        model_kwargs={'device': DEVICE}
    )

    print("🔧 Criando splitters...")

    parent_splitter = RecursiveCharacterTextSplitter(
        chunk_size=PARENT_CHUNK_SIZE,
        chunk_overlap=PARENT_CHUNK_OVERLAP,
        add_start_index=True
    )

    child_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHILD_CHUNK_SIZE,
        chunk_overlap=CHILD_CHUNK_OVERLAP
    )

    print("🔧 Inicializando FAISS...")

    vectorstore = FAISS.from_texts(
        texts=["_INIT_"],
        embedding=embeddings
    )
    vectorstore.delete(list(vectorstore.index_to_docstore_id.values()))

    store = InMemoryStore()

    retriever = ParentDocumentRetriever(
        vectorstore=vectorstore,
        docstore=store,
        parent_splitter=parent_splitter,
        child_splitter=child_splitter
    )

    print("📥 Indexando documentos...")
    retriever.add_documents(documents)

    return vectorstore, store

In [ ]:
def save_index(vectorstore, store):
    print("💾 Salvando FAISS...")
    vectorstore.save_local(VECTORSTORE_PATH)

    print("💾 Salvando docstore...")
    with open(os.path.join(DOCSTORE_PATH, "store.pkl"), "wb") as f:
        pickle.dump(store, f)

    print("✅ Index salvo com sucesso")

In [ ]:
def save_config():
    config = {
        "embedding_model": MODEL_EMBEDDING_NAME,
        "parent_chunk_size": PARENT_CHUNK_SIZE,
        "parent_overlap": PARENT_CHUNK_OVERLAP,
        "child_chunk_size": CHILD_CHUNK_SIZE,
        "child_overlap": CHILD_CHUNK_OVERLAP
    }

    with open("index_config.json", "w") as f:
        json.dump(config, f, indent=4)

    print("✅ Config salva")

In [ ]:
def main():
    print("\n" + "="*60)
    print("🚀 BUILD INDEX")
    print("="*60)

    cleanup_old_index(VECTORSTORE_PATH, DOCSTORE_PATH)

    docs = load_and_clean_definitively(DOCS_PATH)

    df_pages = build_page_count_table(docs)

    print("\n📊 Distribuição de páginas:")
    print(df_pages)

    df_pages.to_csv("distribuicao_paginas.csv")

    vectorstore, store = build_index(docs)

    save_index(vectorstore, store)

    save_config()

    print("\n✅ PROCESSO FINALIZADO COM SUCESSO")

if __name__ == "__main__":
    main()